# QRShield v2 — 3/4: EfficientNetB0 Baselines and Leave-One-Family-Out

| Method | Training data | Notes |
|---|---|---|
| `v1` | plain benign + 2 tamper families, no camera layer | **Reproduction of the v1 setting, not the original v1 weights** |
| `aug` | same as `v1` | adds blur, brightness/contrast, noise, JPEG and affine jitter (augmentation-only baseline) |
| `v2` | full v2 dataset (all families, camera layer) | no extra augmentation |

`v1` and `aug` are *not* retrained for LOFO: three of the five families (sticker, logo, occlusion) are unseen by construction, so their final evaluation on those families is already an unseen-family test. `v2` is retrained once per held-out family.

**Training recipe.** ImageNet-pretrained EfficientNetB0; head first (3 epochs), then the top 30 layers (8 epochs; BatchNorm frozen); best-validation-AUC weights restored. LOFO uses 2 + 5 epochs.

**Session safety.** One job = one result file on Drive. If the session dies, re-run the queue cell; finished jobs are skipped. Memory is released after every job. **Run the benchmark cell before the queue** to see the real time per epoch for your GPU. All timing numbers in this study are estimates until measured.

In [ ]:
# ── Common setup (identical in all four notebooks) ───────────────────────────
import os, gc, sys, json, time, glob, shutil, subprocess, warnings, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running on Colab

SMOKE = False         # False = FULL study (default). True = quick pipeline check on a small slice (separate output folders).
SMOKE_ROWS = 3000     # number of manifest rows used in smoke mode
SEEDS = [0, 1, 2]
IMG = 224
FAMILIES = ['sticker_overlay', 'module_corruption', 'patch_overlay', 'logo_manipulation', 'occlusion_warp']
V1_FAMILIES = ['module_corruption', 'patch_overlay']        # the two tamper families v1 was trained on
METHODS = ['v1', 'aug', 'v2', 'freq', 'dino']

ROOT = Path(os.environ.get('QR_ROOT', '/content/drive/MyDrive/qrshield_v2'))
D = Path(os.environ.get('QR_DATA', '/content/qrshield_v2_data'))          # local copy of the dataset
ZIPS = Path(os.environ.get('QR_ZIPS', '/content/zips'))
OUT = ROOT / ('v2_eval_smoke' if SMOKE else 'v2_eval')
FEA, RES, MOD, TAB = (OUT / n for n in ('feats', 'results', 'models', 'tables'))
for p in (FEA, RES, MOD, TAB):
    p.mkdir(parents=True, exist_ok=True)
print('MODE:', 'SMOKE (small slice, NOT the real study)' if SMOKE else 'FULL study', f'| outputs -> {OUT}')

Mounted at /content/drive
MODE: FULL study | outputs -> /content/drive/MyDrive/qrshield_v2/v2_eval


In [ ]:
# ── TensorFlow ────────────────────────────────────────────────────────────────
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
GPUS = tf.config.list_physical_devices('GPU')
print('GPU:', GPUS)
assert GPUS or SMOKE, 'Select a GPU runtime (Runtime > Change runtime type) for the full study.'

# (head epochs, fine-tune epochs)
EPOCHS = dict(base=(3, 8), final=(3, 8), lofo=(2, 5))
if SMOKE:
    EPOCHS = dict(base=(1, 1), final=(1, 1), lofo=(1, 1))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!rm -rf /content/zips /content/qrshield_v2_data

In [ ]:
# ── Data: copy shards to the local disk, extract, merge the manifest ─────────
def prepare_data():
    if (D / '.done').exists():
        return
    D.mkdir(parents=True, exist_ok=True)
    ZIPS.mkdir(parents=True, exist_ok=True)
    src = sorted(glob.glob(str(ROOT / 'shard500' / 'shard_*.zip')))
    assert src, f'No shard_*.zip found in {ROOT / "shard500"}'
    for z in src:                                   # bulk copy Drive -> local disk (fast)
        dst = ZIPS / Path(z).name
        if not dst.exists():
            shutil.copy(z, dst)
    zips = sorted(glob.glob(str(ZIPS / 'shard_*.zip')))
    def unz(z):
        subprocess.run(['unzip', '-q', '-o', z, '-x', 'manifest.csv', '-d', str(D)], check=True)
    with ThreadPoolExecutor(2) as ex:
        list(ex.map(unz, zips))
    parts = []
    for z in zips:
        with zipfile.ZipFile(z) as zz:
            parts.append(pd.read_csv(zz.open('manifest.csv'), keep_default_na=False))
    merged = pd.concat(parts, ignore_index=True)
    merged.to_csv(D / 'manifest.csv', index=False)
    shutil.rmtree(ZIPS, ignore_errors=True)         # free local disk
    (D / '.done').write_text('ok')
    print('shards:', len(zips), '| manifest rows:', len(merged))

prepare_data()

man = pd.read_csv(D / 'manifest.csv', keep_default_na=False)
if SMOKE:
    man = man.iloc[:SMOKE_ROWS].copy()
need = ['image', 'set', 'split', 'base_id', 'cls', 'family', 'camera']
miss = [c for c in need if c not in man.columns]
assert not miss, f'manifest is missing columns {miss}; found {man.columns.tolist()}'

man['fp'] = str(D) + '/' + man['image'].astype(str)
man['cls_'] = man['cls'].astype(str)
man['label'] = (~man.cls_.str.startswith('benign')).astype(int)
man['fam'] = np.where(man.label == 1, man.cls_.str.replace('^tamper_', '', regex=True), 'benign')
man['base'] = man['base_id'].astype(str)
man['split_'] = man['split'].astype(str)
man['set_'] = man['set'].astype(str)
man['has_cam'] = man['camera'].astype(str).str.strip().str.lower().eq('true')
man['fidx'] = np.arange(len(man))

missing = [p for p in man.fp if not os.path.exists(p)]
assert not missing, f'{len(missing)} image files missing, e.g. {missing[:3]}'
assert set(man.fam.unique()) - {'benign'} == set(FAMILIES), man.fam.unique()

main = man[man.set_ == 'main']
tr = main[main.split_ == 'train']
va = main[main.split_ == 'val']
te = main[main.split_ == 'test']
cam = man[man.set_ == 'camera_bench']
# integrity checks: group split (no base QR in two splits); benchmark only from test bases
assert not set(tr.base) & set(te.base) and not set(tr.base) & set(va.base) and not set(va.base) & set(te.base)
assert not set(cam.base) & set(tr.base) and not set(cam.base) & set(va.base)
share = main.groupby('label').has_cam.mean()
assert 0.3 < share.min() and share.max() < 0.5, f'camera-layer share should be ~0.4 for both labels: {share.to_dict()}'
print(f'rows={len(man)} | main={len(main)} (train {len(tr)} / val {len(va)} / test {len(te)}) | camera_bench={len(cam)}')
print('camera-layer share by label:', share.round(3).to_dict())
if not SMOKE:   # safeguard: the full study must see the whole dataset
    assert len(man) > 60000 and len(tr) > 35000, f'FULL mode expects ~66,000 rows / ~42,000 train images, got {len(man)} / {len(tr)}. Check that all 24 shards are in {ROOT / "shard500"}.'

shards: 24 | manifest rows: 66083
rows=66083 | main=60000 (train 41958 / val 5810 / test 12232) | camera_bench=6083
camera-layer share by label: {0: 0.404, 1: 0.401}


In [ ]:
# ── Metrics and result storage ────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, balanced_accuracy_score

def pick_th(y, p):
    """Decision threshold chosen on the validation set (maximises balanced accuracy)."""
    ths = np.linspace(0.02, 0.98, 97)
    return float(ths[int(np.argmax([balanced_accuracy_score(y, p > t) for t in ths]))])

def m_row(y, p, th):
    y = np.asarray(y)
    return dict(n=int(len(y)), acc=accuracy_score(y, p > th), bal_acc=balanced_accuracy_score(y, p > th),
                auroc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
                f1=f1_score(y, p > th, zero_division=0), acc05=accuracy_score(y, p > 0.5))

def family_records(df, p, th, test, meta):
    """One record for ALL images plus one per tamper family. For a family, the benign images are
    subsampled (fixed seed) to the same size, so accuracy / F1 are not distorted by class imbalance."""
    df = df.reset_index(drop=True)
    ben = np.where(df.label.values == 0)[0]
    groups = [('ALL', np.arange(len(df)))]
    for f in sorted(df.fam[df.label == 1].unique()):
        idx = np.where(df.fam.values == f)[0]
        b = np.random.default_rng(123).choice(ben, size=min(len(idx), len(ben)), replace=False)
        groups.append((f, np.concatenate([b, idx])))
    return [dict(meta, test=test, family=f, th=th, **m_row(df.label.values[i], p[i], th)) for f, i in groups]

def rpath(key): return RES / f'{key}.json'
def done(key): return rpath(key).exists()

def finish(key, score, th, tests, meta):
    recs = []
    for name, df in tests.items():
        recs += family_records(df, score(df), th, name, meta)
    tmp = RES / f'{key}.tmp'
    json.dump(recs, open(tmp, 'w'))
    os.replace(tmp, rpath(key))                      # atomic: a half-written result never counts as done

def tests_final():
    return {'indist': te, 'camera_bench': cam}

def tests_lofo(held):
    return {'indist_heldout': pd.concat([te[te.label == 0], te[te.fam == held]]),
            'camera_heldout': pd.concat([cam[cam.label == 0], cam[cam.fam == held]])}

def v1_subset(df):
    """v1-style training data: plain benign + the two v1 tamper families, no camera layer."""
    return df[(df.fam.isin(V1_FAMILIES) | (df.cls_ == 'benign_plain')) & ~df.has_cam]

def show_status():
    names = [f.stem for f in RES.glob('*__s*.json')]
    print('finished jobs per method:', pd.Series([n.split('__')[0] for n in names]).value_counts().to_dict())

## Input pipeline and model

In [ ]:
# ── Input pipeline, model, training ───────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def _gblur(img):
    s = tf.random.uniform([], 0.5, 2.0)
    k = tf.range(-2., 3.)
    g = tf.exp(-(k ** 2) / (2. * s * s)); g = g / tf.reduce_sum(g)
    k2 = tf.tile((g[:, None] * g[None, :])[:, :, None, None], [1, 1, 3, 1])
    return tf.nn.depthwise_conv2d(img[None], k2, [1, 1, 1, 1], 'SAME')[0]

def aug_fn(img, y):
    """Augmentation-only baseline: blur, brightness/contrast, sensor noise, JPEG compression."""
    img.set_shape([IMG, IMG, 3])
    img = tf.cond(tf.random.uniform([]) < 0.3, lambda: _gblur(img), lambda: img)
    img = tf.image.random_brightness(img, 25.)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = img + tf.random.normal(tf.shape(img), 0., tf.random.uniform([], 0., 8.))
    img = tf.cast(tf.clip_by_value(img, 0., 255.), tf.uint8)
    img = tf.image.random_jpeg_quality(img, 40, 95)
    return tf.cast(img, tf.float32), y

def make_ds(df, train=False, aug=False, bs=64, seed=0):
    ds = tf.data.Dataset.from_tensor_slices((df.fp.values, df.label.values.astype('float32')))
    if train:
        ds = ds.shuffle(min(len(df), 5000), seed=seed)
    def load(p, l):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (IMG, IMG), antialias=True)       # float32, range 0-255
        img.set_shape([IMG, IMG, 3])
        return img, l[None]
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if aug:
        ds = ds.map(aug_fn, num_parallel_calls=AUTOTUNE)
    return ds.batch(bs).prefetch(AUTOTUNE)

def build_eff(geo_aug=False):
    L = tf.keras.layers
    inp = tf.keras.Input((IMG, IMG, 3)); x = inp
    if geo_aug:   # affine approximation of perspective changes (Keras has no perspective layer)
        x = tf.keras.Sequential([L.RandomRotation(0.03, fill_mode='reflect'),
                                 L.RandomZoom(0.1, fill_mode='reflect'),
                                 L.RandomTranslation(0.05, 0.05, fill_mode='reflect')])(x)
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    base.trainable = False
    h = base(x, training=False)                                      # BatchNorm always in inference mode
    h = L.GlobalAveragePooling2D()(h)
    h = L.Dropout(0.3)(h)
    return tf.keras.Model(inp, L.Dense(1, activation='sigmoid')(h)), base

class BestWeights(tf.keras.callbacks.Callback):
    """Restores the weights of the epoch with the best validation AUC."""
    def __init__(self):
        super().__init__(); self.best, self.w = -1, None
    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get('val_auc', -1)
        if v > self.best:
            self.best, self.w = v, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.w is not None:
            self.model.set_weights(self.w)

def compile_model(model, lr):
    model.compile(tf.keras.optimizers.Adam(lr), 'binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

def unfreeze_top(base, n=30):
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, tf.keras.layers.BatchNormalization):
            l.trainable = False

def train_eff(tr_df, va_df, seed, aug, ep):
    tf.keras.utils.set_random_seed(seed)
    model, base = build_eff(geo_aug=aug)
    dtr, dva = make_ds(tr_df, True, aug, seed=seed), make_ds(va_df)
    compile_model(model, 1e-3)                                       # phase 1: classifier head only
    model.fit(dtr, validation_data=dva, epochs=ep[0], callbacks=[BestWeights()], verbose=2)
    unfreeze_top(base)                                               # phase 2: fine-tune the top layers
    compile_model(model, 1e-4)
    model.fit(dtr, validation_data=dva, epochs=ep[1], callbacks=[BestWeights()], verbose=2)
    return model

## Benchmark (run before the queue)

In [ ]:
# ── Throughput benchmark: run this BEFORE the queue ───────────────────────────
def benchmark(n_batches=20, bs=64):
    sub = tr.sample(min(len(tr), n_batches * bs), random_state=0)
    t0 = time.time(); k = 0
    for _ in make_ds(sub, True, bs=bs):
        k += 1
    pipe = (time.time() - t0) / max(k, 1)
    model, base = build_eff(False)
    x, y = next(iter(make_ds(sub.head(bs), bs=bs)))
    unfreeze_top(base); compile_model(model, 1e-4)
    model.train_on_batch(x, y)                                       # warm-up (graph tracing)
    t0 = time.time()
    for _ in range(10):
        model.train_on_batch(x, y)
    step = (time.time() - t0) / 10
    steps_full = len(tr) / bs
    per_epoch = steps_full * max(pipe, step) / 60
    print(f'input pipeline: {pipe:.2f} s/batch | fine-tune step: {step:.2f} s/batch')
    print(f'-> ~{per_epoch:.1f} min per epoch on the full training set (bottleneck: {"input pipeline" if pipe > step else "GPU"})')
    n_v1 = len(v1_subset(tr)) / len(tr)
    ep_full = sum(EPOCHS['final']) + (sum(EPOCHS['base']) * n_v1) * 2 + 5 * sum(EPOCHS['lofo']) * 0.8
    print(f'-> rough total for ONE seed: ~{ep_full * per_epoch / 60:.1f} h (estimate from 10 steps; real time can differ)')
    del model; tf.keras.backend.clear_session(); gc.collect()

benchmark()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
input pipeline: 0.18 s/batch | fine-tune step: 0.06 s/batch
-> ~1.9 min per epoch on the full training set (bottleneck: input pipeline)
-> rough total for ONE seed: ~1.4 h (estimate from 10 steps; real time can differ)


## Training queue, 3 seeds
Order: seed 0 (v2 final, v1, aug, 5 LOFO runs), then seed 1, then seed 2. You can stop after seed 1 and continue later; the tables in notebook 4 work with any number of finished seeds.

In [ ]:
# ── Jobs ──────────────────────────────────────────────────────────────────────
def eff_job(method, setting, seed):
    key = f'{method}__{setting}__s{seed}'
    if done(key):
        return
    held = setting[5:] if setting.startswith('lofo_') else None
    if method in ('v1', 'aug'):                       # v1-style data, with (aug) or without augmentation
        tr_, va_, ep, aug = v1_subset(tr), v1_subset(va), EPOCHS['base'], method == 'aug'
    elif held is None:                                # v2: full dataset, all tamper families
        tr_, va_, ep, aug = tr, va, EPOCHS['final'], False
    else:                                             # v2 leave-one-family-out
        tr_, va_, ep, aug = tr[tr.fam != held], va[va.fam != held], EPOCHS['lofo'], False
    print(f'[{key}] train={len(tr_)} val={len(va_)}', flush=True)
    model = train_eff(tr_, va_, seed, aug, ep)
    score = lambda df: model.predict(make_ds(df), verbose=0).ravel()
    th = pick_th(va_.label.values, score(va_))
    if held is None:                                  # keep final models (adversarial test, re-evaluation)
        model.save(str(MOD / f'{method}_s{seed}.keras'))
        json.dump(dict(th=th), open(MOD / f'{method}_s{seed}.json', 'w'))
    tests = tests_final() if held is None else tests_lofo(held)
    finish(key, score, th, tests, dict(method=method, setting=setting, seed=seed, n_train=len(tr_)))
    del model
    tf.keras.backend.clear_session(); gc.collect()

def job_list():
    J = []
    for seed in SEEDS:                                # seed 0 completes first, then 1, then 2
        J += [('v2', 'final', seed), ('v1', 'final', seed), ('aug', 'final', seed)]
        J += [('v2', f'lofo_{f}', seed) for f in FAMILIES]
    return J

def run_queue(max_minutes=240):
    t0 = time.time()
    todo = [j for j in job_list() if not done(f'{j[0]}__{j[1]}__s{j[2]}')]
    print(f'{len(todo)} of {len(job_list())} jobs remaining')
    for j in todo:
        if (time.time() - t0) / 60 > max_minutes:
            print('time budget reached; run this cell again to continue'); return
        t1 = time.time()
        eff_job(*j)
        print(f'  done {j} ({(time.time() - t1) / 60:.1f} min)', flush=True)
    print('queue complete')

run_queue()
show_status()

8 of 24 jobs remaining
[v2__final__s2] train=41958 val=5810
Epoch 1/3
656/656 - 108s - 165ms/step - accuracy: 0.7333 - auc: 0.8027 - loss: 0.5370 - val_accuracy: 0.7862 - val_auc: 0.8680 - val_loss: 0.4723
Epoch 2/3
656/656 - 42s - 63ms/step - accuracy: 0.7704 - auc: 0.8428 - loss: 0.4792 - val_accuracy: 0.7867 - val_auc: 0.8850 - val_loss: 0.4589
Epoch 3/3
656/656 - 42s - 64ms/step - accuracy: 0.7801 - auc: 0.8519 - loss: 0.4640 - val_accuracy: 0.8103 - val_auc: 0.8908 - val_loss: 0.4217
Epoch 1/8
656/656 - 96s - 146ms/step - accuracy: 0.8401 - auc: 0.9097 - loss: 0.3572 - val_accuracy: 0.8898 - val_auc: 0.9466 - val_loss: 0.2739
Epoch 2/8
656/656 - 48s - 74ms/step - accuracy: 0.8867 - auc: 0.9431 - loss: 0.2776 - val_accuracy: 0.9114 - val_auc: 0.9585 - val_loss: 0.2337
Epoch 3/8
656/656 - 49s - 75ms/step - accuracy: 0.9075 - auc: 0.9548 - loss: 0.2401 - val_accuracy: 0.9201 - val_auc: 0.9614 - val_loss: 0.2163
Epoch 4/8
656/656 - 50s - 76ms/step - accuracy: 0.9187 - auc: 0.9614 - lo